In [ ]:
from pathlib import Path

import pandas as pd
import wbgapi as wb

## 1. World Bank country lookup table

In [ ]:
world_bank_regions: pd.DataFrame = wb.economy.DataFrame()
world_bank_regions.info()

In [ ]:
world_bank_regions

In [ ]:
wb_regions_df = (
    world_bank_regions
    .reset_index()
    .rename(columns={'id': 'iso-alpha3_code'})
    .drop(columns=[
        'aggregate',
        'longitude',
        'latitude',
        'lendingType',
        'incomeLevel',
        'capitalCity'
    ])
    .replace({'': pd.NA})
)
wb_regions_df

In [ ]:
wb_regions_df.info()

## 2. UN M49 country lookup table

In [ ]:
project_dir = Path.cwd().parent
common_table_dir = project_dir / 'data' / 'common-tables'
un_countries_path = common_table_dir / 'UNSD — Methodology.csv'

un_countries_df: pd.DataFrame = pd.read_csv(un_countries_path, sep=';')
original_un_countries_df = un_countries_df.copy(deep=True)

In [ ]:
un_countries_df.head()

In [ ]:
un_countries_df.info()

In [ ]:
un_region_codes_df = (
    un_countries_df
    .filter(
        items=['M49 Code', 'ISO-alpha2 Code', 'ISO-alpha3 Code'],
        axis='columns'
    )
    .rename(columns={'M49 Code': 'm49_code',
                     'ISO-alpha2 Code': 'iso-alpha2_code',
                     'ISO-alpha3 Code': 'iso-alpha3_code'})
)
un_region_codes_df.head(3)

In [ ]:
un_levels: list[tuple] = [
    ('Global Code', 'Global Name'),
    ('Region Code', 'Region Name'),
    ('Sub-region Code', 'Sub-region Name'),
    ('Intermediate Region Code', 'Intermediate Region Name'),
    ('M49 Code', 'Country or Area'),
]

level_frames: list[pd.DataFrame] = []
for code_col, name_col in un_levels:
    temp = (
        un_countries_df
        .dropna(subset=[code_col])
        [[code_col, name_col]]  # Remove all cols except Code & Name
        .rename(columns={code_col: 'm49_code', name_col: 'm49_name'})
    )
    level_frames.append(temp)

# Construct country lookup table
all_regions_df = (
    pd
    .concat(level_frames, ignore_index=True)
    .drop_duplicates(subset=['m49_code', 'm49_name'])
    .sort_values(by='m49_code')
    .reset_index(drop=True)
    .convert_dtypes()
)

# Merge m49 codes with ISO-alpha codes & World Bank names
all_regions_df = all_regions_df.merge(un_region_codes_df, on='m49_code', how='left')
all_regions_df = (
    all_regions_df
    .merge(wb_regions_df, on='iso-alpha3_code', how='outer')
    .rename(columns={'name': 'world_bank_name'})
    .drop(columns=['region', 'adminregion'])
)

In [ ]:
all_regions_df.columns

### Import StatCan regions from `StatCan-transformations.ipynb`

In [ ]:
%store -r statcan_regions

# Convert `statcan_regions` to a dataframe
statcan_regions_df: pd.DataFrame = statcan_regions.rename('statcan_name').to_frame()
original_statcan_regions_df = statcan_regions_df.copy(deep=True)
statcan_regions_df.head()

In [ ]:
un_merged = statcan_regions_df.merge(
    all_regions_df,
    how='left',
    left_on='statcan_name',
    right_on='m49_name',
)
wb_merged = statcan_regions_df.merge(
    all_regions_df,
    how='left',
    left_on='statcan_name',
    right_on='world_bank_name',
)

# Coalesce: combine where `un_merge` is Null
statcan_merged = (
    un_merged
    .combine_first(wb_merged)
    .convert_dtypes()
)
statcan_merged

In [ ]:
statcan_unmatched = (
    statcan_merged.loc[
        statcan_merged[['m49_name', 'world_bank_name']].isna().all(axis=1)
        & statcan_merged['statcan_name'].notna()
    ]
)
statcan_unmatched.head()

In [ ]:
wb_names = set(all_regions_df['world_bank_name'].dropna())
un_names = set(all_regions_df['m49_name'].dropna())
statcan_unmatched_names = set(statcan_unmatched['statcan_name'].dropna())

to_be_matched = {
    'statcan_unmatched': sorted(list(statcan_unmatched_names)),
    'un_names': sorted(list(un_names)),
    'world_bank_names': sorted(list(wb_names))
}
to_be_matched

In [ ]:
# Matches made by ChatGPT o3, then manually reviewed
# Ambiguous matches moved to 'possible matches'
matched_by_ai = {
    "matches": [
        { "statcan": "All countries", "un": "World", "world_bank": "World" },
        { "statcan": "Burma", "un": "Myanmar", "world_bank": "Myanmar" },
        { "statcan": "Central America, South America, and Caribbean", "un": "Latin America and the Caribbean", "world_bank": "Latin America & Caribbean" },
        { "statcan": "Congo, Democratic Republic of the", "un": "Democratic Republic of the Congo", "world_bank": "Congo, Dem. Rep." },
        { "statcan": "Côte d'Ivoire", "un": "Côte d’Ivoire", "world_bank": "Cote d'Ivoire" },
        { "statcan": "Federated States of Micronesia", "un": "Micronesia (Federated States of)", "world_bank": "Micronesia, Fed. Sts." },
        { "statcan": "Hong Kong", "un": "China, Hong Kong Special Administrative Region", "world_bank": "Hong Kong SAR, China" },
        { "statcan": "Iran", "un": "Iran (Islamic Republic of)", "world_bank": "Iran, Islamic Rep." },
        { "statcan": "Laos", "un": "Lao People's Democratic Republic", "world_bank": "Lao PDR" },
        { "statcan": "Macao", "un": "China, Macao Special Administrative Region", "world_bank": "Macao SAR, China" },
        { "statcan": "Netherlands Antilles", "un": "Curaçao", "world_bank": "Curacao" },
        { "statcan": "North Korea", "un": "Democratic People's Republic of Korea", "world_bank": "Korea, Dem. People's Rep." },
        { "statcan": "Saint Martin (French part)", "un": "Saint Martin (French Part)", "world_bank": "St. Martin (French part)" },
        { "statcan": "South Korea", "un": "Republic of Korea", "world_bank": "Korea, Rep." },
        { "statcan": "Syria", "un": "Syrian Arab Republic", "world_bank": "Syrian Arab Republic" },
        { "statcan": "Tanzania, United Republic of", "un": "United Republic of Tanzania", "world_bank": "Tanzania" },
        { "statcan": "Venezuela", "un": "Venezuela (Bolivarian Republic of)", "world_bank": "Venezuela, RB" },
        { "statcan": "Wallis and Futuna", "un": "Wallis and Futuna Islands" },
        { "statcan": "Zair", "un": "Democratic Republic of the Congo", "world_bank": "Congo, Dem. Rep." },
    ],
    "possible_matches": [
        { "statcan": "Middle East", "un": "Western Asia", "world_bank": "Middle East, North Africa, Afghanistan & Pakistan" },
        { "statcan": "Pacific Islands", "un": "Oceania", "world_bank": "Pacific island small states" },
        { "statcan": "South and Central America", "un": "South America", "world_bank": "Latin America & Caribbean" },
        { "statcan": "Other Caribbean", "un": "Caribbean", "world_bank": "Caribbean small states" },
        { "statcan": "Other Europe", "un": "Europe", "world_bank": "Europe & Central Asia" },
        { "statcan": "Other South and Central America", "un": "South America", "world_bank": "Latin America & Caribbean" }
    ],
    "statcan_only": [
        "Asia/Oceania",
        "Czechoslovakia",
        "German Democratic Republic (East)",
        "High Seas",
        "Oceania and Antarctica",
        "Other Africa",
        "Other Asia and Oceania",
        "Unallocated countries",
        "Union of Soviet Socialist Republics",
        "Unspecified",
        "Taiwan",
        "Yugoslavia"
    ]
}

# Create df of matches
un_matches_by_ai = []
statcan_matched_by_ai = []
for match in matched_by_ai['matches']:
    for key, value in match.items():
        if key == 'statcan':
            statcan_matched_by_ai.append(value)
        elif key == 'un':
            un_matches_by_ai.append(value)
  
matched_by_ai_df = pd.DataFrame({'statcan_name': statcan_matched_by_ai,
                                 'm49_name': un_matches_by_ai})

# Create df of StatCan-exclusive regions
statcan_exclusive = []
for possible_match in matched_by_ai['possible_matches']:
    for key, value in possible_match.items():
        if key == 'statcan':
            statcan_exclusive.append(value)

statcan_exclusive_df = pd.DataFrame({'statcan_name': statcan_exclusive})

In [ ]:
# statcan_matched_names = list(
#     set(statcan_regions_df['statcan_name'].to_list())
#     - set(statcan_matched_by_ai)
# )
# statcan_matched_df = pd.DataFrame({'statcan_name': statcan_matched_names})

In [ ]:
# # Insert surrogate key once all transformations are complete
# all_regions_df.insert(0, 'region_id', all_regions_df.index+1)

# # Finally, reorder columns to final positions
# all_codes_df_col_order = [
#     'region_id',
#     'iso-alpha3_code',
#     'iso-alpha2_code',
#     'm49_code',
#     'm49_name',
#     'world_bank_name',
#     'statcan_name
# ]
# all_regions_df = all_regions_df[all_codes_df_col_order]

# all_regions_df.head()

### Aside: UN and World Bank have mutually exclusive ISO country codes

In [ ]:
un_exclusive_iso_codes = (
    set(un_region_codes_df['iso-alpha3_code']) - set(wb_regions_df['iso-alpha3_code'])
)
wb_exclusive_iso_codes = (
    set(wb_regions_df['iso-alpha3_code']) - set(un_region_codes_df['iso-alpha3_code'])
)

exclusive_iso_codes = {
    'code_type': 'iso-alpha3_code',
    'wb_exclusive_codes': sorted(list(wb_exclusive_iso_codes)),
    'un_exclusive_codes': sorted(list(un_exclusive_iso_codes))
}

print(f'UN-exclusive ISO-Alpha 3 codes: {len(un_exclusive_iso_codes)}')
print(f'World Bank-exclusive ISO-Alpha 3 codes: {len(wb_exclusive_iso_codes)}')

In [ ]:


print(f'    World Bank name count: {len(wb_names)}\n'
      f'United Nations name count: {len(un_names)}')

wb_exclusive_names = wb_names - un_names
un_exclusive_names = un_names - wb_names

exclusive_names = {
    'wb_exclusive_names': sorted(list(wb_exclusive_names)),
    'un_exclusive_names': sorted(list(un_exclusive_names))
}
exclusive_names